In [43]:

import torch
from torch import nn, optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import random_split
from datetime import datetime
import numpy as np



# Getting the same results with train and train_manual_update
- Write torch.manual_seed(42) at the beginning of your notebook.
- Write torch.set_default_dtype(torch.double) at the beginning of your notebook to alleviate precision errors

In [44]:
torch.manual_seed(42)
torch.set_default_dtype(torch.double)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



# Tasks
Load, analyse and preprocess the CIFAR-10 dataset. Split it into 3
datasets: training, validation and test. Take a subset of these datasets
by keeping only 2 labels: bird and plane

In [45]:
def load_cifar(train_val_split=0.9, data_path='../data/', preprocessor=None):
    #let's do the preprocessing here, define the transform
    if preprocessor is None:
        cifar10_mean = (0.4914, 0.4822, 0.4465)
        cifar10_std = (0.2023, 0.1994, 0.2010)
        preprocessor = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(cifar10_mean, cifar10_std),
        ])
    train = datasets.CIFAR10(data_path, train=True, download=True, transform=preprocessor)
    test = datasets.CIFAR10(data_path, train=False, download=True, transform=preprocessor)

    keep = {0: 0, 2: 1} #we want to keep only the classes 0 and 2, and we want to remap them to 0 and 1
    #load the dataset, the full cifar10
    def filter_dataset(ds):
        targets = np.array(ds.targets)
        mask = np.isin(targets, [0, 2])
        ds.data = ds.data[mask] #this filteres the images
        ds.targets = [keep[int(t)] for t in targets[mask]] #this filters the labels
    
    filter_dataset(train)
    filter_dataset(test)

    #now we split the training set into train and validation
    n_total = len(train)
    n_train = int(train_val_split * n_total)
    n_val = n_total - n_train

    g = torch.Generator().manual_seed(42)
    train_ds, val_ds = random_split(train, [n_train, n_val], generator=g)

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=64, shuffle=False)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=64, shuffle=False)
    test_loader = torch.utils.data.DataLoader(test, batch_size=64, shuffle=False)
    
    return train_loader, val_loader, test_loader


def compute_accuracy(model, loader):
    model.eval()

    correct = 0
    total = 0
    device = next(model.parameters()).device #either cpu or cuda

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()



    return correct/total 

Implementing a MLP with input dimensions: 3072 (=32-32-3) and the output dimension = 2
Hidden units: 512, 128, 32
Activation function is ReLU

In [46]:
class MyNet(nn.Module):
    def __init__(self):
        super(MyNet, self).__init__()
        self.flatten = nn.Flatten() #we need the 3D images to be flattened into 1D vectors
        self.model = nn.Sequential(
            nn.Linear(32*32*3, 512),
            nn.ReLU(),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 32), #no activiation function here
        )
    def forward(self, x):
        x = self.flatten(x)
        x = self.model(x)
        return x


Write a train(n_epochs, optimizer, model, loss_fn, train_loader) function that trains model for n_epochs epochs given an optimizer optimizer, a loss function loss_fn and a dataloader train_loader.

In [47]:
def train(n_epochs, optimizer, model, loss_fn, train_loader):
    
    device = next(model.parameters()).device
    train_losses = []

    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0.0

        for images, labels in train_loader:
            #move the data to the same device as the model
            images, labels = images.to(device=device, dtype=torch.double), labels.to(device=device, dtype=torch.long)

            optimizer.zero_grad()
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * images.size(0)

        epoch_loss /= len(train_loader.dataset)
        train_losses.append(epoch_loss)
    return train_losses


Write a similar function train manual_update that has no optimizer parameter, but a learning rate lr parameter instead and that manually updates each trainable parameter of model using equation (2). Do not forget to zero out all gradients after each iteration. 

Train 2 instances of MyMLP, one using train and the other using train_manual_update (use the same parameter values for both models). Compare their respective training losses. To get exactly the same results with both functions, see section 3.3

In [48]:
def train_manual_update(n_epochs, model, loss_fn, train_loader, lr=1e-2, momentum_coeff=0., weight_decay=0.):
    train_losses = []
    device = next(model.parameters()).device

    params = [p for p in model.parameters() if p.requires_grad]
    velocity = [torch.zeros_like(p.data) for p in params]
    first_step = True

    for epoch in range(n_epochs):
        model.train()
        running_loss, n_samples = 0.0, 0

        for images, labels in train_loader:
            images = images.to(device=device, dtype=torch.double)
            labels = labels.to(device=device, dtype=torch.long)

            outputs = model(images)
            loss = loss_fn(outputs, labels)

            model.zero_grad()
            loss.backward()

            with torch.no_grad():
                if momentum_coeff == 0.0 and weight_decay == 0.0:
                    # exact plain SGD
                    for p in params:
                        if p.grad is not None:
                            p.data.add_(p.grad, alpha=-lr)
                else:
                    # momentum / weight decay version
                    for p, v in zip(params, velocity):
                        grad = p.grad
                        if weight_decay != 0.0:
                            grad = grad + weight_decay * p.data
                        if momentum_coeff != 0.0:
                            if first_step:
                                v.copy_(grad)
                            else:
                                v.mul_(momentum_coeff).add_(grad)
                            p.data.add_(v, alpha=-lr)
                        else:
                            p.data.add_(grad, alpha=-lr)
            first_step = False                    

            running_loss += loss.item() * images.size(0)
            n_samples += images.size(0)

        train_losses.append(running_loss / n_samples)

    return train_losses


Comparing training loss for both train and train manual

In [49]:
seed = 42
torch.manual_seed(seed)
model_opt = MyNet().to(device).double()

torch.manual_seed(seed)
model_manual= MyNet().to(device).double()


In [50]:
train_loader, val_loader, test_loader = load_cifar()
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model_opt.parameters(), lr=0.1, momentum = 0.9, weight_decay=0.0)
losses = train(10, optimizer, model_opt, loss_fn, train_loader)
losses_manual = train_manual_update(10, model_manual, loss_fn, train_loader, lr=0.1, momentum_coeff=0.9, weight_decay=0.0)
print("Final training loss with optimizer:", losses[-1])
print("Final training loss with manual update:", losses_manual[-1])

#checking that they are equal (should be true if everything matches)
print("Are the final losses equal?", np.isclose(losses[-1], losses_manual[-1]))

for i, (a, b) in enumerate(zip(losses, losses_manual)):
    print(f"Epoch {i+1}: diff = {abs(a-b)}")



Files already downloaded and verified
Files already downloaded and verified
Final training loss with optimizer: 0.6970332927298432
Final training loss with manual update: 0.6970332927298432
Are the final losses equal? True
Epoch 1: diff = 0.0
Epoch 2: diff = 0.0
Epoch 3: diff = 0.0
Epoch 4: diff = 0.0
Epoch 5: diff = 0.0
Epoch 6: diff = 0.0
Epoch 7: diff = 0.0
Epoch 8: diff = 0.0
Epoch 9: diff = 0.0
Epoch 10: diff = 0.0
